# Campaign Analysis

Build campaign-level analytical metrics from the Silver layer to evaluate campaign reach, coupon redemption activity, and customer purchasing behavior.

**Silver Layer**  
↓  
Campaign Definitions  
↓  
Targeted Households  
↓  
Coupon Redemptions  
↓  
Customer Purchase Behavior  
↓  
**Campaign Performance Gold Table**

**Target Grain:** One row per campaign.

In [0]:
%sql
SELECT *
FROM workspace.consumer_analytics_silver.campaign_desc
ORDER BY CAMPAIGN;

DESCRIPTION,CAMPAIGN,START_DAY,END_DAY,_source_file,_ingested_at,_processed_at
TypeB,1,346,383,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeB,2,351,383,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeC,3,356,412,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeB,4,372,404,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeB,5,377,411,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeC,6,393,425,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeB,7,398,432,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeA,8,412,460,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeB,9,435,467,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z
TypeB,10,463,495,campaign_desc.csv,2026-09-23T09:07:56.758Z,2026-09-23T09:18:29.792Z


### 3. Build Campaign Reach Metrics

Aggregate campaign assignments to campaign level to measure the number of households targeted by each campaign.

The resulting dataset follows the target grain of **one row per campaign**.

In [0]:
%sql
SELECT
    CAMPAIGN,
    COUNT(DISTINCT household_key) AS households_assigned

FROM workspace.consumer_analytics_silver.campaign_households

GROUP BY CAMPAIGN

ORDER BY households_assigned DESC;

CAMPAIGN,households_assigned
18,1133
13,1077
8,1076
30,361
26,332
22,276
20,244
14,224
11,214
17,202


In [0]:
%sql
SELECT
    *

FROM workspace.consumer_analytics_silver.campaign_households



DESCRIPTION,household_key,CAMPAIGN,_source_file,_ingested_at,_processed_at
TypeA,17,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,27,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,212,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,208,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,192,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,187,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,183,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,142,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,140,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z
TypeA,134,26,campaign_table.csv,2026-09-23T09:07:59.381Z,2026-09-23T09:18:32.898Z


### 4. Build Campaign Redemption Metrics

Aggregate coupon redemption activity to campaign level to measure customer response.

The metrics capture:
- households that redeemed at least one coupon,
- total coupon redemption events,
- distinct coupons redeemed.

**Target Grain:** One row per campaign.

In [0]:
%sql
SELECT
    CAMPAIGN,

    COUNT(DISTINCT household_key) AS households_redeemed,

    COUNT(*) AS redemption_events,

    COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed

FROM workspace.consumer_analytics_silver.coupon_redemptions

GROUP BY CAMPAIGN

ORDER BY households_redeemed DESC;

CAMPAIGN,households_redeemed,redemption_events,distinct_coupons_redeemed
18,214,653,121
13,196,629,129
8,158,372,105
30,36,64,41
26,31,73,53
25,24,61,15
23,23,60,16
9,20,43,14
20,20,33,15
16,19,43,10


### 5. Combine Campaign Assignment and Redemption Metrics

Combine campaign assignment and coupon redemption metrics to measure campaign-level coupon response.

`redemption_rate` represents the percentage of assigned households that redeemed at least one coupon associated with the campaign.

**Target Grain:** One row per campaign.

In [0]:
%sql
WITH campaign_assignments AS (

    SELECT
        CAMPAIGN,
        COUNT(DISTINCT household_key) AS households_assigned

    FROM workspace.consumer_analytics_silver.campaign_households

    GROUP BY CAMPAIGN
),

campaign_redemptions AS (

    SELECT
        CAMPAIGN,
        COUNT(DISTINCT household_key) AS households_redeemed,
        COUNT(*) AS redemption_events,
        COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed

    FROM workspace.consumer_analytics_silver.coupon_redemptions

    GROUP BY CAMPAIGN
)

SELECT
    a.CAMPAIGN,
    a.households_assigned,

    COALESCE(r.households_redeemed, 0) AS households_redeemed,
    COALESCE(r.redemption_events, 0) AS redemption_events,
    COALESCE(r.distinct_coupons_redeemed, 0) AS distinct_coupons_redeemed,

    ROUND(
        100.0 * COALESCE(r.households_redeemed, 0)
        / a.households_assigned,
        2
    ) AS redemption_rate_pct

FROM campaign_assignments a

LEFT JOIN campaign_redemptions r
    ON a.CAMPAIGN = r.CAMPAIGN

ORDER BY redemption_rate_pct DESC;

CAMPAIGN,households_assigned,households_redeemed,redemption_events,distinct_coupons_redeemed,redemption_rate_pct
18,1133,214,653,121,18.89
13,1077,196,629,129,18.20
3,12,2,2,2,16.67
8,1076,158,372,105,14.68
25,187,24,61,15,12.83
23,183,23,60,16,12.57
15,17,2,2,2,11.76
19,130,15,29,9,11.54
9,176,20,43,14,11.36
29,118,13,24,17,11.02


### 6. Add Campaign Details

Enrich campaign performance metrics with campaign description and campaign duration.

Campaign metadata is joined from the Silver campaign description table while preserving the target grain of **one row per campaign**.

In [0]:
WITH campaign_assignments AS (

    SELECT
        CAMPAIGN,
        COUNT(DISTINCT household_key) AS households_assigned

    FROM workspace.consumer_analytics_silver.campaign_households
    GROUP BY CAMPAIGN
),

campaign_redemptions AS (

    SELECT
        CAMPAIGN,
        COUNT(DISTINCT household_key) AS households_redeemed,
        COUNT(*) AS redemption_events,
        COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed

    FROM workspace.consumer_analytics_silver.coupon_redemptions
    GROUP BY CAMPAIGN
)

SELECT
    d.CAMPAIGN,
    d.DESCRIPTION,
    d.START_DAY,
    d.END_DAY,
    d.END_DAY - d.START_DAY + 1 AS campaign_duration_days,

    a.households_assigned,

    COALESCE(r.households_redeemed, 0) AS households_redeemed,
    COALESCE(r.redemption_events, 0) AS redemption_events,
    COALESCE(r.distinct_coupons_redeemed, 0) AS distinct_coupons_redeemed,

    ROUND(
        100.0 * COALESCE(r.households_redeemed, 0)
        / a.households_assigned,
        2
    ) AS redemption_rate_pct

FROM workspace.consumer_analytics_silver.campaign_desc d

LEFT JOIN campaign_assignments a
    ON d.CAMPAIGN = a.CAMPAIGN

LEFT JOIN campaign_redemptions r
    ON d.CAMPAIGN = r.CAMPAIGN

ORDER BY redemption_rate_pct DESC;

CAMPAIGN,DESCRIPTION,START_DAY,END_DAY,campaign_duration_days,households_assigned,households_redeemed,redemption_events,distinct_coupons_redeemed,redemption_rate_pct
18,TypeA,587,642,56,1133,214,653,121,18.89
13,TypeA,504,551,48,1077,196,629,129,18.20
3,TypeC,356,412,57,12,2,2,2,16.67
8,TypeA,412,460,49,1076,158,372,105,14.68
25,TypeB,659,691,33,187,24,61,15,12.83
23,TypeB,646,684,39,183,23,60,16,12.57
15,TypeC,547,708,162,17,2,2,2,11.76
19,TypeB,603,635,33,130,15,29,9,11.54
9,TypeB,435,467,33,176,20,43,14,11.36
29,TypeB,281,334,54,118,13,24,17,11.02


### 7. Build Campaign Customer Behavior Metrics

Combine campaign assignments with the Customer 360 Gold table to summarize the purchasing behavior of households assigned to each campaign.

This demonstrates reuse of the Customer 360 Gold data product for downstream campaign analytics.

In [0]:
SELECT
    ch.CAMPAIGN,

    ROUND(AVG(c.total_spend), 2) AS avg_customer_spend,

    ROUND(AVG(c.total_baskets), 2) AS avg_customer_baskets,

    ROUND(AVG(c.avg_basket_value), 2) AS avg_customer_basket_value,

    ROUND(AVG(c.active_days), 2) AS avg_customer_active_days

FROM workspace.consumer_analytics_silver.campaign_households ch

INNER JOIN workspace.consumer_analytics_gold.customer_360 c
    ON ch.household_key = c.household_key

GROUP BY ch.CAMPAIGN

ORDER BY avg_customer_spend DESC;

CAMPAIGN,avg_customer_spend,avg_customer_baskets,avg_customer_basket_value,avg_customer_active_days
25,8813.35,263.11,42.37,192.23
6,8751.34,227.48,42.71,177.83
23,8429.33,231.19,44.63,176.81
3,8220.77,267.42,34.9,209.92
21,8168.04,190.26,49.93,153.35
27,8160.07,214.42,49.45,168.58
28,7874.55,175.76,49.19,150.71
24,7754.73,246.31,39.2,180.94
14,7746.4,225.23,41.13,171.7
16,7630.04,227.06,40.05,172.14


### 8. Build Campaign Performance Gold Table

Combine campaign definitions, assignment metrics, coupon redemption metrics, and assigned customer profiles into a single business-ready campaign-level data product.

**Target Grain:** One row per campaign.

In [0]:
CREATE OR REPLACE TABLE workspace.consumer_analytics_gold.campaign_performance
USING DELTA
AS

WITH campaign_assignments AS (

    SELECT
        CAMPAIGN,
        COUNT(DISTINCT household_key) AS households_assigned

    FROM workspace.consumer_analytics_silver.campaign_households

    GROUP BY CAMPAIGN
),

campaign_redemptions AS (

    SELECT
        CAMPAIGN,
        COUNT(DISTINCT household_key) AS households_redeemed,
        COUNT(*) AS redemption_events,
        COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed

    FROM workspace.consumer_analytics_silver.coupon_redemptions

    GROUP BY CAMPAIGN
),

customer_profile AS (

    SELECT
        ch.CAMPAIGN,
        ROUND(AVG(c.total_spend), 2) AS avg_customer_spend,
        ROUND(AVG(c.total_baskets), 2) AS avg_customer_baskets,
        ROUND(AVG(c.avg_basket_value), 2) AS avg_customer_basket_value,
        ROUND(AVG(c.active_days), 2) AS avg_customer_active_days

    FROM workspace.consumer_analytics_silver.campaign_households ch

    INNER JOIN workspace.consumer_analytics_gold.customer_360 c
        ON ch.household_key = c.household_key

    GROUP BY ch.CAMPAIGN
)

SELECT
    d.CAMPAIGN,
    d.DESCRIPTION,
    d.START_DAY,
    d.END_DAY,
    d.END_DAY - d.START_DAY + 1 AS campaign_duration_days,

    COALESCE(a.households_assigned, 0) AS households_assigned,

    COALESCE(r.households_redeemed, 0) AS households_redeemed,
    COALESCE(r.redemption_events, 0) AS redemption_events,
    COALESCE(r.distinct_coupons_redeemed, 0) AS distinct_coupons_redeemed,

    ROUND(
        100.0 * COALESCE(r.households_redeemed, 0)
        / NULLIF(a.households_assigned, 0),
        2
    ) AS redemption_rate_pct,

    p.avg_customer_spend,
    p.avg_customer_baskets,
    p.avg_customer_basket_value,
    p.avg_customer_active_days,

    CURRENT_TIMESTAMP() AS _created_at

FROM workspace.consumer_analytics_silver.campaign_desc d

LEFT JOIN campaign_assignments a
    ON d.CAMPAIGN = a.CAMPAIGN

LEFT JOIN campaign_redemptions r
    ON d.CAMPAIGN = r.CAMPAIGN

LEFT JOIN customer_profile p
    ON d.CAMPAIGN = p.CAMPAIGN;

num_affected_rows,num_inserted_rows


### 9. Validate Campaign Performance

Validate that the final Gold table preserves the expected campaign-level grain.

**Expected Grain:** One row per campaign.

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CAMPAIGN) AS distinct_campaigns
FROM workspace.consumer_analytics_gold.campaign_performance;

total_rows,distinct_campaigns
30,30


### 10. Validate Campaign Metrics

Check the final Gold table for missing metrics and invalid campaign response values.

In [0]:
SELECT
    SUM(CASE WHEN households_assigned IS NULL THEN 1 ELSE 0 END)
        AS null_households_assigned,

    SUM(CASE WHEN households_redeemed IS NULL THEN 1 ELSE 0 END)
        AS null_households_redeemed,

    SUM(CASE WHEN redemption_rate_pct IS NULL THEN 1 ELSE 0 END)
        AS null_redemption_rate,

    SUM(CASE WHEN households_redeemed > households_assigned THEN 1 ELSE 0 END)
        AS redeemed_greater_than_assigned,

    SUM(CASE WHEN redemption_rate_pct < 0 OR redemption_rate_pct > 100 THEN 1 ELSE 0 END)
        AS invalid_redemption_rate

FROM workspace.consumer_analytics_gold.campaign_performance;

null_households_assigned,null_households_redeemed,null_redemption_rate,redeemed_greater_than_assigned,invalid_redemption_rate
0,0,0,0,0
